# GPT desde cero · 1. El tokenizador

Un modelo como GPT no lee letras ni palabras: lee **tokens**. En este notebook programamos desde cero
un tokenizador **BPE** (*Byte Pair Encoding*), lo entrenamos con el Quijote completo y, al final,
lo comparamos con los tokenizadores de verdad de GPT-2, GPT-4 y GPT-4o.

Cada sección corresponde a un capítulo del video. Solo hace falta Python 3; la última sección
usa la librería `tiktoken`, que se instala en la celda correspondiente.

**Cómo usarlo:** ejecuta las celdas en orden, de arriba hacia abajo (en Colab: *Entorno de ejecución →
Ejecutar todas*). El entrenamiento del capítulo 3 tarda alrededor de un minuto.

## Preparación

Importamos lo que vamos a usar y traemos el Quijote de Project Gutenberg (es de dominio público).
El archivo trae un encabezado y un pie en inglés: nos quedamos solo con el texto del libro.

In [ ]:
import re
from collections import Counter
from pathlib import Path
from urllib.request import urlretrieve

# el Quijote, de Project Gutenberg (dominio público)
URL = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
if not Path("quijote.txt").exists():
    urlretrieve(URL, "quijote.txt")
crudo = open("quijote.txt", encoding="utf-8").read()
texto = crudo[crudo.index("*** START"):crudo.index("*** END")]
texto = texto[texto.index("\n") + 1:]   # sin el encabezado

## 1. ¿Letras o palabras?

Antes de hacer cualquier cosa, un modelo tiene que partir el texto en piezas y darle un número a cada una.
Probamos los dos extremos.

**Letras:** `set(texto)` se queda con cada letra una sola vez. Su largo es el tamaño del vocabulario;
`len(texto)`, cuántas piezas tendría el texto.

In [ ]:
letras = set(texto)
print(len(letras), len(texto))

**Palabras:** `re.findall(r"\w+", texto)` busca todos los pedazos que son "una o más letras seguidas"
(`\w` = letra, número o guion bajo).

In [ ]:
palabras = re.findall(r"\w+", texto)
print(len(set(palabras)), len(palabras))

`Counter` es un diccionario que cuenta cuántas veces aparece cada cosa. ¿Cuántas palabras aparecen
**una sola vez**? Un modelo casi no tendría ejemplos para aprenderlas.

In [ ]:
cuantas = Counter(palabras)
una_vez = [p for p, n in cuantas.items() if n == 1]
print(len(una_vez))

Las letras dan un vocabulario chico pero un texto larguísimo; las palabras, un texto corto pero un
vocabulario enorme (y cualquier palabra nueva queda afuera). Necesitamos algo en el medio.

## 2. La idea de BPE

BPE (de 1994, pensado para comprimir archivos) arma ese "algo en el medio" con un bucle muy simple:

1. **contar** cuántas veces aparece cada par de piezas vecinas;
2. **fusionar** el par más frecuente en una pieza nueva;
3. **repetir**.

En cada ronda el vocabulario crece en una pieza y el texto se acorta. BPE no sabe nada de gramática:
junta lo que es frecuente.

## 3. Programamos BPE

**Contar pares.** `piezas[1:]` es la misma lista corrida un lugar; `zip` recorre las dos a la vez,
así que cada pieza queda junto a la que le sigue.

In [ ]:
def contar_pares(piezas):
    pares = Counter()
    for a, b in zip(piezas, piezas[1:]):
        pares[a, b] += 1
    return pares

**Fusionar.** Recorremos la lista con un índice `i`. `piezas[i:i + 2]` son las dos piezas que empiezan
en `i`: si son el par, agregamos la pieza nueva y saltamos dos lugares; si no, copiamos una y avanzamos uno.

In [ ]:
def fusionar(piezas, par, nueva):
    resultado = []
    i = 0
    while i < len(piezas):
        if piezas[i:i + 2] == list(par):
            resultado.append(nueva)
            i += 2
        else:
            resultado.append(piezas[i])
            i += 1
    return resultado

**Una prueba.** Con una frase corta, cuatro rondas. Fíjate en la tercera: junta pedazos de dos palabras distintas.

In [ ]:
piezas = list("la mar, la marea")
for _ in range(4):
    par = contar_pares(piezas).most_common(1)[0][0]
    piezas = fusionar(piezas, par, par[0] + par[1])
    print(par, piezas)

**Primero, trozos.** Como GPT, cortamos el texto en trozos con una expresión regular (un espacio opcional
seguido de letras; o un espacio opcional seguido de signos; o espacios) y las fusiones nunca cruzan de un
trozo a otro. De paso ganamos velocidad: contamos cada trozo distinto **una sola vez** y, al contar pares,
sumamos `veces` (cuántas veces aparece ese trozo) en lugar de uno.

In [ ]:
PATRON = r" ?\w+| ?[^\w\s]+|\s+"
trozos = Counter(re.findall(PATRON, texto))
partes = {t: list(t) for t in trozos}

def contar_pares(partes, trozos):
    pares = Counter()
    for t, veces in trozos.items():
        p = partes[t]
        for a, b in zip(p, p[1:]):
            pares[a, b] += veces
    return pares

**Entrenar.** 500 rondas de contar y fusionar. La lista `fusiones`, en orden, **es** el tokenizador.
(Tarda alrededor de un minuto.)

In [ ]:
fusiones = []
for _ in range(500):
    par = contar_pares(partes, trozos).most_common(1)[0][0]
    for t in partes:
        partes[t] = fusionar(partes[t], par, par[0] + par[1])
    fusiones.append(par)
print([a + b for a, b in fusiones[:10]])

¿En qué fusión aparecen los personajes? (El espacio de adelante es parte del token.)

In [ ]:
nuevas = [a + b for a, b in fusiones]
for palabra in [" Quijote", " Sancho", " caballero"]:
    print(repr(palabra), nuevas.index(palabra) + 1)

## 4. Usar el tokenizador

Para cortar un texto nuevo hacemos lo mismo que al entrenar, pero sin contar: cortamos en trozos,
partimos cada trozo en letras y aplicamos las fusiones **en el mismo orden** en que se aprendieron.

In [ ]:
def codificar(texto):
    piezas = []
    for t in re.findall(PATRON, texto):
        p = list(t)
        for par in fusiones:
            p = fusionar(p, par, par[0] + par[1])
        piezas += p
    return piezas

frase = "Don Quijote le mandó un mensaje a Sancho por el celular."
print(codificar(frase))

Nunca hay una palabra imposible: en el peor caso, sale letra por letra. Y ojo con el espacio: " Sancho"
(con el espacio adelante) es un token, pero "Sancho" al principio de una frase es otro trozo y se parte.

In [ ]:
print(codificar(" grzlp"))
print(codificar(" Sancho"), codificar("Sancho"))

A cada pieza le damos un número (su lugar en el vocabulario): eso es lo que de verdad entra al modelo.
Volver atrás es pegar las piezas.

In [ ]:
vocab = sorted(letras) + [a + b for a, b in fusiones]
numero = {pieza: i for i, pieza in enumerate(vocab)}
print([numero[p] for p in codificar(frase)])
print("".join(codificar(frase)) == frase)

## 5. Bytes, no letras

Nuestro tokenizador arranca con las letras del Quijote: un emoji o una letra en chino no están en su lista,
así que no tendría con qué escribirlos.

In [ ]:
print("🙂" in numero, "中" in numero)

GPT arranca un nivel más abajo, en los **bytes**: todo texto se guarda como bytes y solo hay 256 posibles.
`.encode("utf-8")` convierte un texto en sus bytes y `list` los muestra como números del 0 al 255.

In [ ]:
print(list("a".encode("utf-8")))
print(list("ñ".encode("utf-8")))
print(list("🙂".encode("utf-8")))

## 6. El tokenizador de verdad

OpenAI publica sus tokenizadores en la librería `tiktoken`. La siguiente celda la instala (en Colab hace
falta; si ya la tienes, no hace nada).

In [ ]:
%pip install -q tiktoken

`get_encoding` carga un tokenizador por su nombre: `gpt2` (el de GPT-2 y GPT-3), `cl100k_base` (GPT-4)
y `o200k_base` (GPT-4o). `n_vocab` es el tamaño de su vocabulario.

In [ ]:
import tiktoken

for nombre in ["gpt2", "cl100k_base", "o200k_base"]:
    print(nombre, tiktoken.get_encoding(nombre).n_vocab)
print("el nuestro", len(vocab))

Cómo corta cada uno algunas palabras. Fíjate en " Quijote": en el nuestro es un solo token.

In [ ]:
for nombre in ["gpt2", "cl100k_base", "o200k_base"]:
    enc = tiktoken.get_encoding(nombre)
    for palabra in [" Quijote", "murciélago", "ñandú"]:
        piezas = [enc.decode([n]) for n in enc.encode(palabra)]
        print(nombre, repr(palabra), piezas)

El mismo texto en español y en inglés (artículo 1 de la Declaración Universal de Derechos Humanos,
en sus versiones oficiales). `encode` devuelve la lista de números: contamos cuántos tokens usa cada uno.

In [ ]:
es = ("Todos los seres humanos nacen libres e iguales "
      "en dignidad y derechos y, dotados como están de "
      "razón y conciencia, deben comportarse "
      "fraternalmente los unos con los otros.")
en = ("All human beings are born free and equal in "
      "dignity and rights. They are endowed with reason "
      "and conscience and should act towards one another "
      "in a spirit of brotherhood.")
for nombre in ["gpt2", "cl100k_base", "o200k_base"]:
    enc = tiktoken.get_encoding(nombre)
    print(nombre, len(enc.encode(es)), len(enc.encode(en)))

## Resumen

- Un modelo no lee letras ni palabras: lee **tokens**.
- BPE los arma juntando, una y otra vez, el par de piezas vecinas más frecuente.
- El texto con el que se entrena el tokenizador decide qué idiomas salen "baratos" (menos tokens) y cuáles "caros".

En el próximo video de **GPT desde cero**, esos números se convierten en vectores y programamos nuestro
primer modelo que aprende a escribir.